# 06 Calibration & Explainability

Validation avancée du modèle final entraîné dans le notebook 05:
- calibration globale et par segments métier
- impact du seuil opérationnel
- lecture des variables explicatives sauvegardées

In [7]:
%pip install -q pandas numpy scikit-learn joblib
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path('..').resolve()))
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_score, recall_score, f1_score

ROOT = pathlib.Path('..').resolve()
MODELS_DIR = ROOT / 'models'
PROC_DIR = ROOT / 'data' / 'processed'

artifact = joblib.load(MODELS_DIR / 'logreg_raw.pkl')
model = artifact['model']
feature_cols = artifact['features']

threshold_path = MODELS_DIR / 'best_threshold.txt'
best_threshold = float(threshold_path.read_text().strip()) if threshold_path.exists() else 0.5

train = pd.read_csv(PROC_DIR / 'train_features.csv')
y = train['target_risk_flag'].astype(int)

# Reconstruire les segments métiers utilisés dans le notebook 05
def build_business_segments(df: pd.DataFrame) -> pd.DataFrame:
    seg = pd.DataFrame(index=df.index)

    if 'loanamount' in df.columns:
        seg['seg_loanamount'] = pd.cut(
            df['loanamount'],
            bins=[-np.inf, 10000, 30000, np.inf],
            labels=['small', 'medium', 'large']
        ).astype(str)

    if 'termdays' in df.columns:
        seg['seg_termdays'] = pd.cut(
            df['termdays'],
            bins=[-np.inf, 30, 60, np.inf],
            labels=['short', 'mid', 'long']
        ).astype(str)

    if 'due_vs_loan_ratio' in df.columns:
        seg['seg_pricing_ratio'] = pd.cut(
            df['due_vs_loan_ratio'],
            bins=[-np.inf, 1.20, 1.35, np.inf],
            labels=['low_cost', 'standard', 'high_cost']
        ).astype(str)

    if 'customer_age_approx' in df.columns:
        seg['seg_age'] = pd.cut(
            df['customer_age_approx'],
            bins=[-np.inf, 25, 35, 45, np.inf],
            labels=['young', 'early_career', 'mid_career', 'senior']
        ).astype(str)

    if 'max_late_days' in df.columns:
        seg['seg_late_behavior'] = pd.cut(
            df['max_late_days'].fillna(0),
            bins=[-np.inf, 0, 7, 30, np.inf],
            labels=['on_time', 'minor_delay', 'moderate_delay', 'severe_delay']
        ).astype(str)

    return seg

seg_df = build_business_segments(train)
seg_ohe = pd.get_dummies(seg_df, prefix=seg_df.columns, dummy_na=False)

X_full = pd.concat([train.copy(), seg_ohe], axis=1)
for col in feature_cols:
    if col not in X_full.columns:
        X_full[col] = 0

X = X_full[feature_cols].fillna(0)
proba = model.predict_proba(X)[:, 1]

print(f"Model chargé: {type(model).__name__}")
print(f"Features: {len(feature_cols)}")
print(f"Shape X: {X.shape} | défaut moyen: {y.mean():.2%}")
print(f"Seuil opérationnel: {best_threshold:.4f}")
print(f"ROC-AUC: {roc_auc_score(y, proba):.4f} | Brier: {brier_score_loss(y, proba):.4f}")

Note: you may need to restart the kernel to use updated packages.
Model chargé: CalibratedClassifierCV
Features: 21
Shape X: (4368, 21) | défaut moyen: 21.79%
Seuil opérationnel: 0.2309
ROC-AUC: 0.7179 | Brier: 0.1505


In [8]:
# 1) Calibration globale
fraction_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy='uniform')

print("=== Calibration globale ===")
print(f"{'Score prédit':>14}  {'Défaut réel':>14}  {'Gap':>8}")
print("-" * 45)
for pred, real in zip(mean_pred, fraction_pos):
    gap = real - pred
    print(f"  {pred:12.3f}  {real:12.3f}  {gap:+.3f}")

# 2) Impact du seuil opérationnel
pred_05 = (proba >= 0.5).astype(int)
pred_best = (proba >= best_threshold).astype(int)

print("\n=== Qualité de classification ===")
print(f"@0.5  -> Precision={precision_score(y, pred_05, zero_division=0):.4f} | Recall={recall_score(y, pred_05, zero_division=0):.4f} | F1={f1_score(y, pred_05, zero_division=0):.4f}")
print(f"@best -> Precision={precision_score(y, pred_best, zero_division=0):.4f} | Recall={recall_score(y, pred_best, zero_division=0):.4f} | F1={f1_score(y, pred_best, zero_division=0):.4f}")

# 3) Calibration par segments métier (sur segments reconstruits)
segment_cols = [c for c in X_full.columns if c.startswith('seg_') and c in feature_cols]
print("\n=== Calibration par segment (top 12) ===")
if segment_cols:
    seg_rows = []
    for col in segment_cols:
        for val, grp in X_full.groupby(col):
            if val not in (0, 1):
                continue
            if val != 1:
                continue
            idx = grp.index
            if len(idx) < 80:
                continue
            y_g = y.loc[idx]
            p_g = proba[idx]
            seg_rows.append({
                'segment': col,
                'n': len(idx),
                'default_rate': float(y_g.mean()),
                'avg_score': float(np.mean(p_g)),
                'gap': float(y_g.mean() - np.mean(p_g)),
            })

    seg_df = pd.DataFrame(seg_rows)
    if not seg_df.empty:
        seg_df['abs_gap'] = seg_df['gap'].abs()
        print(seg_df.sort_values('abs_gap', ascending=False).head(12).drop(columns=['abs_gap']).round(4))
    else:
        print('Pas assez de volume par segment pour estimer des gaps stables.')
else:
    print('Aucun segment métier retrouvé dans les variables du modèle.')

# 4) Explainability alignée production: coefficients sauvegardés
coef_path = MODELS_DIR / 'model_coefficients.csv'
if coef_path.exists():
    coef_df = pd.read_csv(coef_path, index_col=0)
    coef_df.columns = ['coef']
    coef_df['abs_coef'] = coef_df['coef'].abs()
    print("\n=== Top 15 variables explicatives (|coef|) ===")
    print(coef_df.sort_values('abs_coef', ascending=False).head(15).drop(columns=['abs_coef']).round(4))
else:
    print("\nmodel_coefficients.csv introuvable. Exécuter d'abord le notebook 05.")

=== Calibration globale ===
  Score prédit     Défaut réel       Gap
---------------------------------------------
         0.073         0.063  -0.010
         0.152         0.145  -0.007
         0.236         0.237  +0.000
         0.348         0.409  +0.062
         0.449         0.466  +0.018
         0.542         0.516  -0.027
         0.642         0.636  -0.005
         0.738         0.719  -0.019
         0.820         0.333  -0.487
         0.973         0.615  -0.357

=== Qualité de classification ===
@0.5  -> Precision=0.5781 | Recall=0.1555 | F1=0.2450
@best -> Precision=0.4224 | Recall=0.5347 | F1=0.4720

=== Calibration par segment (top 12) ===
                            segment     n  default_rate  avg_score     gap
2                  seg_termdays_mid   288        0.2257     0.2622 -0.0365
0              seg_loanamount_large   429        0.1375     0.1648 -0.0272
1             seg_loanamount_medium  1477        0.1611     0.1705 -0.0093
7  seg_late_behavior_moderate_